# Connectivity methods and hyperparameter sensitivity

Keep the baseline sensitivity analyses and the final Figure 4 comparison. The final comparison uses 30 seeds per method and 2 million steps.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from multi_mnist.analysis import (
    asymptotic_metric, filter_sweep, load_export, normalize_columns,
    plot_sensitivity, save_fig, set_style,
)

%matplotlib inline
set_style()

## Sweeps, axes and cells

Each arm spans one or two Comet sweep names: the v2 grid, plus a `v2b` top-up
that re-ran one width after the original sweep lost its remaining assignments.
`filter_sweep` merges them and `dedupe_trials` keeps one trial per cell,
preferring trials that logged every expected period (a divergence counts as
finished, since stopping early on divergence is a completed measurement).
That is what collapses both the top-up overlap and the failed retries.

In [ ]:
PROJECT = 'paper-weight-pruning-connectivity-sweep'

SWEEPS = {
    'SET':           ['04_set_n16_v2', '04_set_n16_v2b_h1024'],
    'DEEP-R':        ['04_deep_r_n16_v2', '04_deep_r_n16_v2b_h768'],
    'Static sparse': ['04_static_sparse_n16_v2'],
    'Dense':         ['04_dense_n16_v2'],
    'Block-sparse':  ['04_block_sparse_n16_v2'],
}

LR = 'optimizer.learning_rate'

# The swept axes of each arm. The learning rate is last everywhere; the arms
# with no other axis are tuned on it alone.
AXES = {
    'SET':           ['target_hidden_units', 'algorithm.evolve_frequency', LR],
    'DEEP-R':        ['target_hidden_units', 'algorithm.l1',
                      'algorithm.noise_ratio', LR],
    'Static sparse': ['target_hidden_units', LR],
    'Dense':         [LR],
    'Block-sparse':  [LR],
}

# How each axis reads on an x scale. The learning-rate grid is powers of two;
# l1 spans decades; the rest are plotted linearly -- widths include 768 and
# noise_ratio includes 0, neither of which survives a log axis.
AXIS_STYLE = {
    LR:                          dict(label='Step-size', log_base=2),
    'target_hidden_units':       dict(label='Hidden units', log_base=None),
    'algorithm.evolve_frequency':dict(label='Evolve frequency (steps)', log_base=None),
    'algorithm.l1':              dict(label=r'$\alpha$  (L1)', log_base=10),
    'algorithm.noise_ratio':     dict(label=r'Noise ratio  $\sqrt{2\eta T}/\eta\alpha$',
                                      log_base=None),
}

FIG_DIR = '../figures/generated'   # output, rewritten on every re-run

TAIL_FRAC = 0.10   # asymptotic = mean over the final 10% of logged periods

In [ ]:
# Run once to fetch from Comet, then leave commented out.
# from multi_mnist.analysis import download_project
# download_project(PROJECT, output_dir='data')

cfg_all, run_all = load_export(PROJECT, data_dir='data')
cfg_all = normalize_columns(cfg_all)
print(f'{len(cfg_all)} trials, {len(run_all)} metric rows')

## One row per grid cell

`asymptotic_metric` averages the final `TAIL_FRAC` of each trial's logged loss,
nulls any trial flagged diverged, and aggregates with `skipna=False` — so a
single bad seed makes the whole cell NaN rather than quietly averaging away.
NaNs are kept from here on. They drop out of the "best over the rest" step
because a NaN can never be the minimum, and a parameter value whose every
alternative is NaN stays NaN and leaves a visible gap in the curve.

In [ ]:
def cell_table(method):
    """Asymptotic loss for every cell of one arm's grid."""
    axes = AXES[method]
    cfg, run = filter_sweep(cfg_all, run_all, SWEEPS[method], cell_cols=axes)
    # asymptotic_metric always groups by the learning rate, so pass the others.
    others = [a for a in axes if a != LR]
    return asymptotic_metric(cfg, run, 'loss', others, tail_frac=TAIL_FRAC)


cells_by_method = {m: cell_table(m) for m in SWEEPS}

for m, tab in cells_by_method.items():
    n_nan = int(tab['_metric'].isna().sum())
    print(f'{m:14s} {len(tab):3d} cells, {n_nan:3d} NaN '
          f'(diverged or a NaN seed), best loss {tab["_metric"].min():.3f}')

## Sensitivity: best-of-the-rest at each value

For each value of the axis, the minimum over every other hyperparameter.

In [ ]:
def sensitivity(method, x_col):
    """Best (lowest) asymptotic loss achievable at each value of ``x_col``."""
    tab = cells_by_method[method]
    # min() skips NaN, so a diverged alternative does not hide a good one;
    # an all-NaN value yields NaN and breaks the line, which is the honest
    # rendering of "nothing at this setting finished".
    out = (tab.groupby(x_col, dropna=False)['_metric']
              .min().reset_index().sort_values(x_col))
    return out


def plot_method(method, fig_width=4.0, ylim=None):
    axes = AXES[method]
    fig, axs = plt.subplots(1, len(axes), figsize=(fig_width * len(axes), 3.4),
                            squeeze=False)
    for ax, col in zip(axs[0], axes):
        style = AXIS_STYLE[col]
        sub = sensitivity(method, col)
        plot_sensitivity(
            sub, ax=ax, x_col=col, y_col='_metric',
            xlabel=style['label'], ylabel='Asymptotic loss',
            log_base=style['log_base'], ylim=ylim)
        if not style['log_base']:
            # plot_sensitivity only pins ticks on a log axis; on a linear one
            # matplotlib would invent its own (300, 400, ...) and hide which
            # values were actually run.
            ticks = sorted(sub[col].dropna().unique())
            ax.set_xticks(ticks)
            ax.set_xticklabels([f'{t:g}' for t in ticks])
    fig.suptitle(method, y=1.02)
    fig.tight_layout()
    return fig, axs

### SET

Three axes: the width the budget is spread over, how often the
prune-and-regrow event fires (ζ is derived from it, holding the paper's
per-example rewiring rate fixed), and the step-size.

In [ ]:
fig, _ = plot_method('SET')
save_fig('04_sensitivity_set', fig_dir=FIG_DIR)

### DEEP-R

Four axes. `l1` is the strength of the pull toward zero that drives pruning,
and `noise_ratio` is the Langevin noise scaled to that pull — 0 turns the noise
off entirely, leaving deterministic sign-flip pruning.

In [ ]:
fig, _ = plot_method('DEEP-R')
save_fig('04_sensitivity_deep_r', fig_dir=FIG_DIR)

### Static sparse — SET's initialization, frozen

In [ ]:
fig, _ = plot_method('Static sparse')
save_fig('04_sensitivity_static_sparse', fig_dir=FIG_DIR)

### The two fixed-connectivity arms

Both are tuned on the step-size alone: block-sparse sits at the width that
defines the shared budget, and dense's width is derived from that budget
(16 units, since a fully connected unit costs 12,704 weights at 16 tasks).
Note the two use different step-size windows — dense diverges above 2⁻⁹.

In [ ]:
for m in ('Block-sparse', 'Dense'):
    fig, _ = plot_method(m, fig_width=4.6)
    save_fig(f'04_sensitivity_{m.lower().replace("-", "_")}', fig_dir=FIG_DIR)

## All arms on one step-size axis

The one axis every arm shares, so the five are directly comparable.

In [ ]:
from multi_mnist.analysis import get_color_palette

fig, ax = plt.subplots(figsize=(5.5, 3.8))
# A class->color mapping, so SET, DEEP-R, dense and block-sparse keep the
# colors they carry in the rest of the paper.
colors = get_color_palette(list(SWEEPS))
for method in SWEEPS:
    sub = sensitivity(method, LR)
    ax.plot(sub[LR], sub['_metric'], '-o', color=colors[method], label=method)
ax.set_xscale('log', base=2)
ticks = sorted(set().union(*[set(sensitivity(m, LR)[LR].dropna()) for m in SWEEPS]))
ax.set_xticks(ticks)
ax.set_xticklabels([f'$2^{{{int(np.log2(v))}}}$' for v in ticks])
ax.set_xlabel('Step-size')
ax.set_ylabel('Asymptotic loss')
ax.grid(True, alpha=0.4)
ax.legend(loc='best')
fig.tight_layout()
save_fig('04_sensitivity_all_step_size', fig_dir=FIG_DIR)

## Best cell per arm

The winning configuration behind each curve's minimum.

In [ ]:
rows = []
for method, axes in AXES.items():
    tab = cells_by_method[method]
    best = tab.loc[tab['_metric'].idxmin()]
    rows.append({'arm': method, 'loss': round(float(best['_metric']), 4),
                 **{a.split('.')[-1]: best[a] for a in axes}})
pd.DataFrame(rows).set_index('arm')

## v3 — SET and DEEP-R at a fixed step-size

v2 left both arms with their best cell on a grid edge, so v3 re-centres the
windows: SET's width moves down to 64-512 and its evolve frequency up to
500-4000, DEEP-R's `l1` extends down to 0. The step-size is fixed at 2^-8,
interior for every arm in v2, and DEEP-R's `noise_ratio` is pinned at the
paper's 1.0 after returning a null at both horizons.

In [ ]:
V3_SWEEPS = {'SET': ['04_set_n16_v3'], 'DEEP-R': ['04_deep_r_n16_v3']}

# The step-size is fixed in v3, so it is not an axis here.
V3_AXES = {
    'SET':    ['target_hidden_units', 'algorithm.evolve_frequency'],
    'DEEP-R': ['target_hidden_units', 'algorithm.l1'],
}


def v3_cell_table(method):
    axes = V3_AXES[method]
    cfg, run = filter_sweep(cfg_all, run_all, V3_SWEEPS[method],
                            cell_cols=axes + [LR])
    return asymptotic_metric(cfg, run, 'loss', axes, tail_frac=TAIL_FRAC)


v3_by_method = {m: v3_cell_table(m) for m in V3_SWEEPS}

for m, tab in v3_by_method.items():
    n_nan = int(tab['_metric'].isna().sum())
    print(f'{m:8s} {len(tab):2d} cells, {n_nan} NaN, best loss {tab["_metric"].min():.3f}')

In [ ]:
def plot_method_v3(method, fig_width=4.0, ylim=None):
    """The same sensitivity view as plot_method, over the v3 grid."""
    axes = V3_AXES[method]
    tab = v3_by_method[method]
    fig, axs = plt.subplots(1, len(axes), figsize=(fig_width * len(axes), 3.4),
                            squeeze=False)
    for ax, col in zip(axs[0], axes):
        style = AXIS_STYLE[col]
        sub = (tab.groupby(col, dropna=False)['_metric']
                  .min().reset_index().sort_values(col))
        if style['log_base']:
            # A log axis cannot show 0, and leaving it in collapses the scale
            # so the decades that matter overlap. l1=0 is reported separately.
            sub = sub[sub[col] > 0]
        plot_sensitivity(
            sub, ax=ax, x_col=col, y_col='_metric',
            xlabel=style['label'], ylabel='Asymptotic loss',
            log_base=style['log_base'], ylim=ylim)
        if not style['log_base']:
            ticks = sorted(sub[col].dropna().unique())
            ax.set_xticks(ticks)
            ax.set_xticklabels([f'{t:g}' for t in ticks])
    fig.suptitle(f'{method}  (v3, step-size $2^{{-8}}$)', y=1.02)
    fig.tight_layout()
    return fig, axs

### SET

In [ ]:
fig, _ = plot_method_v3('SET')
save_fig('04_sensitivity_set_v3', fig_dir=FIG_DIR)

### DEEP-R

`l1` is plotted on a log axis, so the `l1`=0 cell cannot appear on it; its
value is printed below the figure.

In [ ]:
fig, _ = plot_method_v3('DEEP-R')
save_fig('04_sensitivity_deep_r_v3', fig_dir=FIG_DIR)

tab = v3_by_method['DEEP-R']
zero = tab[tab['algorithm.l1'] == 0]['_metric']
if len(zero):
    print(f'l1 = 0 (off the log axis): best asymptotic loss {zero.min():.3f}')

In [ ]:
from multi_mnist.analysis import (
    fetch_per_seed_arrays, get_color_palette, label_curves_inline,
    measure_inline_labels, plot_ci_curve, stack_per_seed,
)

BEST_PROJECT = 'paper-weight-pruning-connectivity-best'
ARM_COLORS = get_color_palette(['Dense', 'Block-sparse', 'Static sparse', 'SET', 'DEEP-R'])
FIXED_CONNECTIVITY = ('Dense', 'Block-sparse', 'Static sparse')
LOG_FREQ = 1000


## Figure 4 — learned versus fixed connectivity

Each method uses 30 distinct seeds and 2 million steps. Continual SET uses median utility resets and random regrowth across each layer (`best_2m/continual_set_median_global.yaml`).

Path purity is `path_purity_all`, the fraction of all input–hidden–output paths whose endpoints belong to the same subproblem. Accuracy uses the final 10% of steps. Duplicate seed batches are excluded.


In [ ]:
SWEEPS_2M = {'Dense': '04_dense_n16_best_2m', 'Block-sparse': '04_block_sparse_n16_best_2m',
             'Static sparse': '04_static_sparse_n16_best_2m', 'SET': '04_set_n16_best_2m',
             'DEEP-R': '04_deep_r_n16_best_2m', 'Continual SET': '04_continual_set_median_global_n16_best_2m'}
colors = {**ARM_COLORS, 'Continual SET': next(
    c for c in get_color_palette() if c not in ARM_COLORS.values())}

# Finished arms: per-seed curves from the export, both halves pooled.
# Refresh the export when needed.
# from multi_mnist.analysis import download_project
# download_project(BEST_PROJECT, output_dir='data')
cfg_2m, run_2m = load_export(BEST_PROJECT, data_dir='data')
cfg_2m = normalize_columns(cfg_2m)
acc_2m, purity_2m = {}, {}
for arm, sweep in SWEEPS_2M.items():
    cfg, run = filter_sweep(cfg_2m, run_2m, [sweep], cell_cols=['seed_offset'])
    assert len(cfg) == 2, f'{arm}: expected two distinct seed batches'
    run_ids = cfg['run_id'].tolist()
    acc_2m[arm] = stack_per_seed(fetch_per_seed_arrays(run_ids, cache_dir='data/per_seed_cache'), run_ids)[1]
    assert acc_2m[arm].shape == (2000, 30), (arm, acc_2m[arm].shape)
    series = [g['path_purity_all'].dropna().to_numpy() for _, g in run.groupby('run_id')]
    purity_2m[arm] = np.mean([s[:min(map(len, series))] for s in series], axis=0)

# Inline labels in the style of the paper's Figure 4: each arm named in its own
# color beside its curve, x and side set here, heights by label_curves_inline.
# The purity panel carries no labels; it shares the accuracy panel's colors.
# An x at the right edge is pulled back inside, so those labels end flush right.
RIGHT = 2_000_000
LABELS_2M = {
    'Block-sparse':  (RIGHT, 'above'),
    'Continual SET': (RIGHT, 'below'),
    'SET':           (RIGHT, 'below'),
    'DEEP-R':        (0.51e6, 'below'),
    'Static sparse': (RIGHT, 'below'),
    'Dense':         (RIGHT, 'below'),
}

set_style('2-col')
fig, ax = plt.subplots()
mean_curves = {}
for arm in SWEEPS_2M:
    if arm in acc_2m:
        steps = np.arange(acc_2m[arm].shape[0]) * LOG_FREQ
        plot_ci_curve(steps, acc_2m[arm], color=colors[arm], ax=ax)
        mean_curves[arm] = (steps, acc_2m[arm].mean(axis=1))
ax.set_xlim(0, 2_000_000)
ax.set_xticks(np.arange(0, 2_000_001, 500_000))
ax.set_ylim(0, 1.0)
ax.set_xlabel('Step')
ax.set_ylabel('Accuracy')
ax.set_title('Learned vs Fixed Connectivity')
ax.grid(True); ax.set_axisbelow(True)
ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
fig.tight_layout()
specs = [({'DEEP-R': 'DEEP R'}.get(arm, arm), x, side, colors[arm], mean_curves[arm])
         for arm, (x, side) in LABELS_2M.items() if arm in mean_curves]
texts = label_curves_inline(ax, specs)
print(measure_inline_labels(ax, specs, texts))
for fmt in ('pdf', 'png'):
    fig.savefig(f'{FIG_DIR}/{fmt}/fig4_2m_accuracy.{fmt}', bbox_inches='tight', dpi=400)
plt.show()

set_style('2-col')
fig, ax = plt.subplots()
for arm in ('Dense', 'SET', 'DEEP-R', 'Continual SET'):
    if arm in FIXED_CONNECTIVITY and arm in purity_2m:
        ax.axhline(float(np.mean(purity_2m[arm])), ls='--', lw=2.4, color=colors[arm])
    elif arm in purity_2m:
        ax.plot(np.arange(len(purity_2m[arm])) * LOG_FREQ, purity_2m[arm], color=colors[arm])
ax.set_xlim(0, 2_000_000)
ax.set_xticks(np.arange(0, 2_000_001, 500_000))
ax.set_ylim(0.06, 0.10)
ax.set_xlabel('Step')
ax.set_ylabel('Path purity')
ax.set_title('Path Purity')
ax.grid(True); ax.set_axisbelow(True)
ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
fig.tight_layout()
for fmt in ('pdf', 'png'):
    fig.savefig(f'{FIG_DIR}/{fmt}/fig4_2m_path_purity.{fmt}', bbox_inches='tight', dpi=400)
plt.show()

for arm in SWEEPS_2M:
    if arm in acc_2m:
        a = acc_2m[arm]
        print(f'{arm:14s} {a.shape[1]} seeds, last-10% accuracy {np.nanmean(a[-int(TAIL_FRAC * len(a)):]):.4f}, '
              f'final path purity {purity_2m[arm][-1]:.4f}')
    elif arm in running:
        print(f'{arm:14s} running, at step {int(running[arm]["accuracy"][-1, 0]):,}')